In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
import csv
from pathlib import Path
import pubchempy as pcp
from loguru import logger
from tqdm.auto import tqdm

import time


In [ ]:
INPUT = require_stage("04_selection") / "selected_ligands.csv"
MAX_ATTEMPTS = 3

In [ ]:
def lookup_pubchem(smiles):
    for attempt in range(MAX_ATTEMPTS):
        try:
            return pcp.get_cids(smiles, namespace="smiles"), None
        except Exception as exc:
            logger.warning("PubChem attempt {} failed: {}", attempt + 1, exc)
            if attempt + 1 == MAX_ATTEMPTS:
                return None, str(exc)
            time.sleep(2 ** attempt)


In [ ]:
with stage_run("05_pubchem", [INPUT], {}) as output_dir:
    with INPUT.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        columns = list(reader.fieldnames or [])
        if not {"Name", "SMILES"} <= set(columns):
            raise ValueError("Missing Name or SMILES column")
        rows = list(reader)
    unmatched = []
    with (output_dir / "pubchem_results.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns + ["PubChemStatus", "PubChemCIDs", "PubChemError"])
        writer.writeheader()
        for row in tqdm(rows, desc="Checking PubChem", unit="mol"):
            cids, error = lookup_pubchem(row.get("CanonicalSMILES") or row["SMILES"])
            status = "ERROR" if error else ("FOUND" if cids else "NOT_FOUND")
            writer.writerow(dict(row, PubChemStatus=status, PubChemCIDs=";".join(map(str, cids or [])), PubChemError=error or ""))
            if status == "NOT_FOUND":
                unmatched.append(row)
    with (output_dir / "pubchem_unmatched_ligands.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(unmatched)
    logger.info("Checked: {} | Not found: {}", len(rows), len(unmatched))
